# Inspect Chroma behavior

@hanoihantrakul 15AUG2024
To perform unified tokenizer training on Vocal Music, Instrumental Music and Speech, it is beneficial to look at the behavior of chroma in these three datasets.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torchaudio
import IPython.display as ipd
assert torch.cuda.is_available()
import sys
import os
import numpy as np
from recipes.umm.transforms.chroma import ChromaSpectrogram

%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
os.chdir('/opt/tiger/samantha')

# Load Dataset

In [ ]:
from recipes.datasets.mcc.mix_mkii import MixDataModule

In [ ]:
SAMPLE_RATE = 24000
HOP_LENGTH = 240
BATCH_SIZE = 8
SHUFFLE_BUFFER_SIZE = 10
MIN_DURATION = 1
MAX_DURATION = 60
TOKENIZER = None
FRAME_RATE = 25

DATA_IDS = [794] # 2255=Vocal Music, 2375=Speech, 794=Instrumentals
DATA_WEIGHTS = [1]

mdm = MixDataModule(
    data_ids=DATA_IDS,
    data_weights=DATA_WEIGHTS,
    batch_size=BATCH_SIZE * MAX_DURATION * SAMPLE_RATE, # means "total number of audio samples loaded into memory"
    shuffle_buffer_size=SHUFFLE_BUFFER_SIZE,
    sample_rate=SAMPLE_RATE,
    min_duration=MIN_DURATION,
    max_duration=MAX_DURATION,
    frame_rate=FRAME_RATE,
    tokenizer=TOKENIZER,
)

# There will be many printouts if loading for the first time

In [ ]:
dm = mdm.train_dataloader()
dm_it = iter(dm)

In [ ]:
NUM_ITERATIONS=5

audio_list = []
token_list = []
for i in range(NUM_ITERATIONS):
    batch = next(dm_it)
    print(batch.keys())
    audio_list.append(batch['audio'])
    token_list.append(batch['token'])

print(len(audio_list))
print(audio_list[0].shape)
print(len(token_list))
print(token_list[0].shape)

# Inspect Chroma

In [ ]:
N_FFT=2048
WIN_LENGTH=2048
HOP_LENGTH=240
N_CHROMA=12
chroma_transform = ChromaSpectrogram(
                sample_rate=SAMPLE_RATE,
                n_fft=N_FFT,
                win_length=WIN_LENGTH,
                hop_length=HOP_LENGTH,
                n_chroma=N_CHROMA,
                normalized=False,
            )

In [ ]:
chroma_spec = chroma_transform(audio_list[0])
print(chroma_spec.shape)

In [ ]:
SAMPLE_IDX = 18
TIME_START = 10
TIME_END = 14 #int(audio_list[0].shape[-1]/SAMPLE_RATE)

In [ ]:
plt.figure(figsize=(16, 8))
plt.pcolor(chroma_spec[SAMPLE_IDX][0].cpu()[:,TIME_START*int((SAMPLE_RATE/HOP_LENGTH)):TIME_END*int((SAMPLE_RATE/HOP_LENGTH))])

In [ ]:
ipd.Audio(audio_list[0][SAMPLE_IDX][:, int(TIME_START*SAMPLE_RATE):int(TIME_END*SAMPLE_RATE)], rate=SAMPLE_RATE)